In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from skimage import io, filters, morphology, measure, segmentation, exposure, util
from skimage.filters import threshold_otsu, sato, meijering
from skimage.morphology import remove_small_objects, remove_small_holes, disk
from scipy import ndimage as ndi
from scipy.stats import mannwhitneyu

# ---- user inputs ----
image_folder = r"" # folder containing all TIFFs (all samples mixed)
pixel_size_um = 0.568182
vessel_neighborhood_um = 5.0
min_nucleus_area_um2 = 25
nuclear_dilate_um = 2.0
min_vessel_object_um2 = 10
use_vesselness = True
vesselness_method = "sato"

x4_threshold_mode = "otsu"
x4_percentile = 99.0
prrx1_threshold_mode = "otsu"
prrx1_percentile = 99.0

pvalue_category_to_show = "X4+PRRX1+"
figure_out = "bv_lv_X4_PRRX1_boxdot.png"
per_field_csv = "bv_lv_X4_PRRX1_per_field_percentages.csv"
per_nucleus_csv = "bv_lv_X4_PRRX1_per_nucleus_allfields.csv"

required_markers = ["Hoechst", "X4", "PRRX1", "CD31", "PROX1"]

def um2_to_px2(area_um2, pix_um):
    return max(1, int(round(area_um2 / (pix_um**2))))

def um_to_px(dist_um, pix_um):
    return max(1, int(round(dist_um / pix_um)))

def find_samples_regions_from_filenames(folder):
    """
    Expect filenames like:
      SampleID_regionX_Hoechst.tif
      SampleID_regionX_X4.tif
    Parses:
      sample_id = first chunk
      region_id = second chunk
      marker    = last chunk
    and builds:
      {sample_id: {region_id: {marker: path, ...}, ...}, ...}
    """
    samples = {}
    for ext in ("*.tif", "*.tiff"):
        for path in glob.glob(os.path.join(folder, ext)):
            base = os.path.splitext(os.path.basename(path))[0]
            parts = base.split("_")
            if len(parts) < 3:
                # not matching pattern, skip
                continue
            sample_id = parts[0]         # e.g. NGMEL4
            region_id = parts[1]         # e.g. region2
            marker    = parts[-1]        # e.g. Hoechst

            if marker not in required_markers:
                continue

            samples.setdefault(sample_id, {})
            samples[sample_id].setdefault(region_id, {})
            samples[sample_id][region_id][marker] = path

    # drop incomplete regions
    complete = {}
    for sample_id, regions in samples.items():
        good_regions = {
            r: d for r, d in regions.items()
            if all(m in d for m in required_markers)
        }
        if good_regions:
            complete[sample_id] = good_regions

    return complete

samples = find_samples_regions_from_filenames(image_folder)

print("Found samples and regions:")
for s, regs in samples.items():
    print(f"  {s}: {sorted(regs.keys())}")
def load_region_channels(paths_dict):
    chans = {}
    for m in required_markers:
        img = io.imread(paths_dict[m])
        img = util.img_as_float32(img)
        if img.ndim != 2:
            raise ValueError(f"{os.path.basename(paths_dict[m])} is not 2D (shape={img.shape})")
        chans[m] = img
    H, W = chans["Hoechst"].shape
    return chans, (H, W)

def segment_nuclei(hoechst, pix_um, min_area_um2, dilate_um):
    ho = filters.gaussian(hoechst, sigma=1)
    ho = exposure.rescale_intensity(ho, in_range="image", out_range=(0,1))
    th = threshold_otsu(ho)
    nuc = ho > th
    nuc = remove_small_objects(nuc, um2_to_px2(min_area_um2, pix_um))
    nuc = morphology.binary_closing(nuc, footprint=disk(1))
    dist = ndi.distance_transform_edt(nuc)
    maxima = morphology.h_maxima(dist, 0.5)
    markers = measure.label(maxima)
    nuc_labels = segmentation.watershed(-dist, markers, mask=nuc)
    nuc_labels_dil = segmentation.expand_labels(nuc_labels, distance=um_to_px(dilate_um, pix_um))
    return nuc_labels, nuc_labels_dil

def segment_vessel_like(channel_img, pix_um, min_obj_um2, use_vn=True, method="sato"):
    im = filters.gaussian(channel_img, sigma=1)
    if use_vn:
        vn = sato(im) if method == "sato" else meijering(im)
        vn = exposure.rescale_intensity(vn, in_range="image", out_range=(0,1))
        t = threshold_otsu(vn)
        m = vn > t
    else:
        t = threshold_otsu(im)
        m = im > t
    m = remove_small_objects(m, um2_to_px2(min_obj_um2, pix_um))
    m = remove_small_holes(m, area_threshold=um2_to_px2(5, pix_um))
    m = morphology.binary_closing(m, footprint=disk(1))
    return m

def per_nucleus_intensity(labels, img):
    props = measure.regionprops_table(
        labels, intensity_image=img,
        properties=("label", "mean_intensity", "centroid")
    )
    return pd.DataFrame(props)

def find_thr(vals, mode="otsu", pct=99.0):
    v = np.asarray(vals, dtype=np.float32)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return 0.0
    if mode == "otsu":
        return float(threshold_otsu(v) if (v > 0).sum() >= 5 else np.percentile(v, 95))
    return float(np.percentile(v, pct))

def classify_and_summarize(sample_id, region_id, paths_dict):
    chans, (H, W) = load_region_channels(paths_dict)
    Hoechst = chans["Hoechst"]
    X4      = chans["X4"]
    PRRX1   = chans["PRRX1"]
    CD31    = chans["CD31"]
    PROX1   = chans["PROX1"]

    # nuclei
    nuc_labels, nuc_labels_dil = segment_nuclei(
        Hoechst, pixel_size_um, min_nucleus_area_um2, nuclear_dilate_um
    )

    # vessels + lymphatics
    cd31_mask  = segment_vessel_like(CD31, pixel_size_um, min_vessel_object_um2,
                                     use_vesselness, vesselness_method)
    prox1_mask = segment_vessel_like(PROX1, pixel_size_um, min_vessel_object_um2,
                                     use_vesselness, vesselness_method)
    vessel_mask = cd31_mask | prox1_mask
    vessel_plus = ndi.binary_dilation(
        vessel_mask, structure=disk(um_to_px(vessel_neighborhood_um, pixel_size_um))
    )

    # per-nucleus signal
    df_x4  = per_nucleus_intensity(nuc_labels_dil, X4)
    df_prr = per_nucleus_intensity(nuc_labels_dil, PRRX1)
    df = df_x4.merge(df_prr[["label", "mean_intensity"]],
                     on="label", suffixes=("_x4", "_prrx1"))

    cy = df["centroid-0"].round().clip(0, H-1).astype(int)
    cx = df["centroid-1"].round().clip(0, W-1).astype(int)
    df["region_type"] = np.where(vessel_plus[cy, cx], "BV/LV+", "BV/LV-")

    thr_x4   = find_thr(df["mean_intensity_x4"],   mode=x4_threshold_mode,   pct=x4_percentile)
    thr_prrx = find_thr(df["mean_intensity_prrx1"], mode=prrx1_threshold_mode, pct=prrx1_percentile)

    df["X4+"]        = df["mean_intensity_x4"]    >= thr_x4
    df["PRRX1+"]     = df["mean_intensity_prrx1"] >= thr_prrx
    df["X4+PRRX1+"]  = df["X4+"] & df["PRRX1+"]
    df["Negative"]   = ~(df["X4+"] | df["PRRX1+"])

    # tag with sample + region
    df["sample"] = sample_id
    df["region"] = region_id

    # per-field %
    out = []
    for rt in ["BV/LV+", "BV/LV-"]:
        sub = df[df["region_type"] == rt]
        n = len(sub)
        pct = {"Negative": np.nan, "X4+": np.nan, "PRRX1+": np.nan, "X4+PRRX1+": np.nan}
        if n:
            pct = {
                "Negative":   100.0 * sub["Negative"].mean(),
                "X4+":        100.0 * sub["X4+"].mean(),
                "PRRX1+":     100.0 * sub["PRRX1+"].mean(),
                "X4+PRRX1+":  100.0 * sub["X4+PRRX1+"].mean()
            }
        pct["sample"]       = sample_id
        pct["region"]       = region_id
        pct["region_type"]  = rt
        pct["nuclei_total"] = n
        out.append(pct)

    return df, pd.DataFrame(out)
all_nuclei, all_fields = [], []

for sample_id, regions in samples.items():
    for region_id, paths_dict in regions.items():
        print(f"Analyzing {sample_id} / {region_id} ...")
        df_nuc, df_field = classify_and_summarize(sample_id, region_id, paths_dict)
        all_nuclei.append(df_nuc)
        all_fields.append(df_field)

df_nuclei = pd.concat(all_nuclei, ignore_index=True)
df_fields = pd.concat(all_fields,  ignore_index=True)

df_nuclei.to_csv(per_nucleus_csv, index=False)
df_fields.to_csv(per_field_csv, index=False)

print("Per-nucleus table:", per_nucleus_csv)
print("Per-field table:",  per_field_csv)




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

cats = ["Negative", "X4+", "PRRX1+", "X4+PRRX1+"]

# dict of values per group+category for boxplots
data = {
    (r, c): df_fields[df_fields["region_type"] == r][c].dropna().values
    for r in ["BV/LV+", "BV/LV-"] for c in cats
}

N_plus  = int(df_fields.loc[df_fields.region_type == "BV/LV+", "nuclei_total"].sum())
N_minus = int(df_fields.loc[df_fields.region_type == "BV/LV-", "nuclei_total"].sum())

fig, ax = plt.subplots(figsize=(8, 6))
x_base_plus  = np.arange(4)
x_base_minus = np.arange(4) + 5
width = 0.6

# background shading
ax.axvspan(-0.5, 3.5, color=(1, 0, 0, 0.08))   # BV/LV+
ax.axvspan(4.5, 8.5, color=(0, 0, 0, 0.06))    # BV/LV-

boxprops    = dict(linewidth=1)
medianprops = dict(linewidth=1.5)
whiskerprops = dict(linewidth=1)
capprops    = dict(linewidth=1)

# boxplots 
ax.boxplot(
    [data[("BV/LV+", c)] for c in cats],
    positions=x_base_plus, widths=width,
    patch_artist=False, boxprops=boxprops,
    medianprops=medianprops, whiskerprops=whiskerprops, capprops=capprops,
)
ax.boxplot(
    [data[("BV/LV-", c)] for c in cats],
    positions=x_base_minus, widths=width,
    patch_artist=False, boxprops=boxprops,
    medianprops=medianprops, whiskerprops=whiskerprops, capprops=capprops,
)

# -------- colored dots per sample -------- #
samples = sorted(df_fields["sample"].unique())
palette = plt.cm.tab10(np.linspace(0, 1, len(samples)))  # 3 nice distinct colors
sample_color = {s: palette[i] for i, s in enumerate(samples)}

rng = np.random.default_rng(123)

for _, row in df_fields.iterrows():
    samp   = row["sample"]
    rtype  = row["region_type"]   # "BV/LV+" or "BV/LV-"
    color  = sample_color[samp]

    for j, c in enumerate(cats):
        y = row[c]
        if np.isnan(y):
            continue
        if rtype == "BV/LV+":
            x0 = x_base_plus[j]
        else:  # BV/LV-
            x0 = x_base_minus[j]
        x = x0 + (rng.random() - 0.5) * 0.4  # jitter
        ax.plot(x, y, 'o', ms=4, alpha=0.9, color=color)

# axis labels etc.
ax.set_xticks(list(x_base_plus) + list(x_base_minus))
ax.set_xticklabels(cats + cats, rotation=30, ha='right')
ax.set_ylabel("% of cells (per region)")
ax.set_xlim(-0.75, 8.75)
ax.set_ylim(-2, 110)

ax.text(1.5, 98, f"BV/LV+ area\nN={N_plus}",  ha='center', va='top', fontsize=10)
ax.text(6.5, 98, f"BV/LV- area\nN={N_minus}", ha='center', va='top', fontsize=10)

# legend for samples
handles = [plt.Line2D([0], [0], marker='o', linestyle='',
                      color=sample_color[s], label=s) for s in samples]
ax.legend(handles=handles, title="Sample", loc="upper right", frameon=False)

# optional p-value for one category
p_cat = "X4+PRRX1+"
if p_cat in cats:
    ci = cats.index(p_cat)
    a = data[("BV/LV+", p_cat)]
    b = data[("BV/LV-", p_cat)]
    if len(a) and len(b):
        stat, p = mannwhitneyu(a, b, alternative='two-sided')
        ptext = "p<0.0001" if p < 1e-4 else f"p={p:.3g}"
        y = max((np.max(a) if len(a) else 0),
                (np.max(b) if len(b) else 0)) + 5

        def add_bracket(ax, x1, x2, y, text):
            h = 3
            ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, c='k')
            ax.text((x1 + x2) / 2, y + h + 1, text,
                    ha='center', va='bottom', fontsize=10)

        add_bracket(ax, x_base_plus[ci], x_base_minus[ci], y, ptext)

plt.tight_layout()
plt.savefig("bv_lv_X4_PRRX1_boxdot_bySample.png", dpi=300)
plt.show()






In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def plot_bv_lv_boxdot(
    df_fields,
    categories=("Negative","X4+","PRRX1+","X4+PRRX1+"),
    pvalue_category="X4+PRRX1+",
    outfile="bv_lv_X4_PRRX1_boxdot_bySample.png",
    figsize=(6.5, 4.0),   # ~1.5-column figure
    dpi=600,
    add_violin=False
):
    cats = list(categories)

    # Data for box/violin
    data = {
        (r, c): df_fields[df_fields["region_type"] == r][c].dropna().values
        for r in ["BV/LV+", "BV/LV-"] for c in cats
    }

    N_plus  = int(df_fields.loc[df_fields.region_type == "BV/LV+", "nuclei_total"].sum())
    N_minus = int(df_fields.loc[df_fields.region_type == "BV/LV-", "nuclei_total"].sum())

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    x_base_plus  = np.arange(len(cats))
    x_base_minus = np.arange(len(cats)) + len(cats) + 1  # gap in between
    width = 0.6

    # Shaded backgrounds
    ax.axvspan(-0.5, len(cats)-0.5, color=(1, 0, 0, 0.08))         # BV/LV+
    ax.axvspan(len(cats)+0.5, 2*len(cats)+0.5, color=(0, 0, 0, 0.06))  # BV/LV-

    # Optional violins (nice but subtle)
    if add_violin:
        for side, base, region in [("left", x_base_plus, "BV/LV+"),
                                   ("right", x_base_minus, "BV/LV-")]:
            for i, c in enumerate(cats):
                vals = data[(region, c)]
                if len(vals) < 2:
                    continue
                vp = ax.violinplot(
                    vals, positions=[base[i]],
                    widths=width,
                    showextrema=False
                )
                for pc in vp['bodies']:
                    pc.set_alpha(0.1)

    # Boxplots
    boxprops    = dict(linewidth=1)
    medianprops = dict(linewidth=1.2)
    whiskerprops = dict(linewidth=1)
    capprops    = dict(linewidth=1)

    ax.boxplot(
        [data[("BV/LV+", c)] for c in cats],
        positions=x_base_plus, widths=width,
        patch_artist=False, boxprops=boxprops,
        medianprops=medianprops, whiskerprops=whiskerprops, capprops=capprops,
    )
    ax.boxplot(
        [data[("BV/LV-", c)] for c in cats],
        positions=x_base_minus, widths=width,
        patch_artist=False, boxprops=boxprops,
        medianprops=medianprops, whiskerprops=whiskerprops, capprops=capprops,
    )

    # Sample-coloured dots
    samples = sorted(df_fields["sample"].unique())
    palette = plt.cm.tab10(np.linspace(0, 1, len(samples)))
    sample_color = {s: palette[i] for i, s in enumerate(samples)}

    rng = np.random.default_rng(123)

    for _, row in df_fields.iterrows():
        samp   = row["sample"]
        rtype  = row["region_type"]   # "BV/LV+" or "BV/LV-"
        color  = sample_color[samp]

        for j, c in enumerate(cats):
            y = row[c]
            if np.isnan(y):
                continue
            if rtype == "BV/LV+":
                x0 = x_base_plus[j]
            else:
                x0 = x_base_minus[j]
            x = x0 + (rng.random() - 0.5) * 0.4  # jitter
            ax.plot(x, y, 'o', ms=4, alpha=0.9, color=color)

    # Axes, labels, limits
    all_y = df_fields[cats].values.ravel()
    all_y = all_y[np.isfinite(all_y)]
    ymax  = max(all_y.max(), 100)
    ax.set_ylim(0, ymax * 1.10)   # give 10% headroom so whiskers don’t clip

    ax.set_xticks(list(x_base_plus) + list(x_base_minus))
    ax.set_xticklabels(cats + cats, rotation=30, ha='right')
    ax.set_ylabel("% of cells (per region)")

    ax.set_xlim(-0.75, 2*len(cats)-0.25)

    ax.text(np.mean(x_base_plus), ymax*1.05, f"BV/LV+ area\nN={N_plus}",
            ha='center', va='top', fontsize=9)
    ax.text(np.mean(x_base_minus), ymax*1.05, f"BV/LV- area\nN={N_minus}",
            ha='center', va='top', fontsize=9)

    # Sample legend
    handles = [plt.Line2D([0], [0], marker='o', linestyle='',
                          color=sample_color[s], label=s)
               for s in samples]
    ax.legend(handles=handles, title="Sample", loc="upper right", frameon=False)

    # P-value annotation for BV/LV+ vs BV/LV- for one category
    if pvalue_category in cats:
        ci = cats.index(pvalue_category)
        a = data[("BV/LV+", pvalue_category)]
        b = data[("BV/LV-", pvalue_category)]

        if len(a) and len(b):
            stat, p = mannwhitneyu(a, b, alternative='two-sided')
            ptext = "p<0.0001" if p < 1e-4 else f"p={p:.3g}"
            y = max(a.max() if len(a) else 0, b.max() if len(b) else 0) + ymax*0.05

            def add_bracket(ax, x1, x2, y, text):
                h = ymax*0.03
                ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, c='k')
                ax.text((x1 + x2) / 2, y + h, text,
                        ha='center', va='bottom', fontsize=9)

            add_bracket(ax, x_base_plus[ci], x_base_minus[ci], y, ptext)

    plt.tight_layout()
    plt.savefig(outfile, dpi=dpi, bbox_inches="tight")
    plt.show()

    print("Saved figure to:", outfile)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def plot_bv_lv_boxdot_no_negative(
    df_fields,
    outfile="bv_lv_X4_PRRX1_noNegative.png",
    figsize=(6.2, 4.0),
    dpi=600,
):
    
    cats = ["X4+", "PRRX1+", "X4+PRRX1+"]

    # Extract data per group
    data = {
        (r, c): df_fields[df_fields["region_type"] == r][c].dropna().values
        for r in ["BV/LV+", "BV/LV-"] for c in cats
    }

    # Total nuclei (annotation)
    N_plus  = int(df_fields.loc[df_fields.region_type == "BV/LV+", "nuclei_total"].sum())
    N_minus = int(df_fields.loc[df_fields.region_type == "BV/LV-", "nuclei_total"].sum())

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # x-locations
    x_base_plus  = np.arange(len(cats))
    x_base_minus = np.arange(len(cats)) + len(cats) + 1  # separation gap
    width = 0.55

    # Shaded backgrounds
    ax.axvspan(-0.5, len(cats)-0.5, color=(1, 0, 0, 0.08))         # BV/LV+
    ax.axvspan(len(cats)+0.5, 2*len(cats)+0.5, color=(0, 0, 0, 0.06))  # BV/LV-

    # Boxplot styling
    boxprops     = dict(linewidth=1)
    medianprops  = dict(linewidth=1.2)
    whiskerprops = dict(linewidth=1)
    capprops     = dict(linewidth=1)

    # Boxplots for both regions
    ax.boxplot(
        [data[("BV/LV+", c)] for c in cats],
        positions=x_base_plus, widths=width,
        boxprops=boxprops, medianprops=medianprops,
        whiskerprops=whiskerprops, capprops=capprops,
        patch_artist=False,
    )

    ax.boxplot(
        [data[("BV/LV-", c)] for c in cats],
        positions=x_base_minus, widths=width,
        boxprops=boxprops, medianprops=medianprops,
        whiskerprops=whiskerprops, capprops=capprops,
        patch_artist=False,
    )

    # UNIQUE COLORS PER SAMPLE
    samples = sorted(df_fields["sample"].unique())
    palette = plt.cm.tab10(np.linspace(0, 1, len(samples)))
    sample_color = {s: palette[i] for i, s in enumerate(samples)}

    rng = np.random.default_rng(123)

    # Colored dots for each region
    for _, row in df_fields.iterrows():
        samp  = row["sample"]
        rtype = row["region_type"]
        col   = sample_color[samp]

        for j, c in enumerate(cats):
            y = row[c]
            if np.isnan(y):
                continue

            x0 = x_base_plus[j] if rtype == "BV/LV+" else x_base_minus[j]
            x  = x0 + (rng.random() - 0.5) * 0.35  # jitter

            ax.plot(x, y, 'o', ms=4.2, alpha=0.9, color=col)

    # Appropriate y-limits based on data
    all_y = df_fields[cats].values.ravel()
    all_y = all_y[np.isfinite(all_y)]
    ymax  = max(all_y.max(), 100)
    ax.set_ylim(0, ymax * 1.10)

    # Labels
    ax.set_xticks(list(x_base_plus) + list(x_base_minus))
    ax.set_xticklabels(cats + cats, rotation=30, ha='right')
    ax.set_ylabel("% of cells (per region)")

    ax.text(np.mean(x_base_plus), ymax*1.06,
            f"BV/LV+ area\nN={N_plus}",
            ha='center', va='top', fontsize=9)

    ax.text(np.mean(x_base_minus), ymax*1.06,
            f"BV/LV- area\nN={N_minus}",
            ha='center', va='top', fontsize=9)

    # Sample legend
    handles = [
        plt.Line2D([0], [0], marker='o', linestyle='',
                   color=sample_color[s], label=s)
        for s in samples
    ]
    ax.legend(handles=handles, title="Sample",
              loc="upper right", frameon=False, fontsize=8)

    # Optional p-value for X4+PRRX1+
    p_cat = "X4+PRRX1+"
    if p_cat in cats:
        a = data[("BV/LV+", p_cat)]
        b = data[("BV/LV-", p_cat)]
        if len(a) and len(b):
            stat, p = mannwhitneyu(a, b, alternative='two-sided')
            ptext = "p<0.0001" if p < 1e-4 else f"p={p:.3g}"
            y = max(a.max(), b.max()) + ymax*0.05

            def add_bracket(ax, x1, x2, y, text):
                h = ymax * 0.03
                ax.plot([x1, x1, x2, x2],
                        [y, y+h, y+h, y], lw=1, c='k')
                ax.text((x1 + x2)/2, y + h, text,
                        ha='center', va='bottom', fontsize=9)

            add_bracket(ax,
                        x_base_plus[cats.index(p_cat)],
                        x_base_minus[cats.index(p_cat)],
                        y, ptext)

    plt.tight_layout()
    plt.savefig(outfile, dpi=dpi, bbox_inches="tight")
    plt.show()

    print("Saved:", outfile)
